# station_pairs table

### In this notebook I will populate the station_pairs table, it has the following columns:



# Imports

In [ ]:
import pandas as pd
import json
import os
import psycopg2
from psycopg2.extras import execute_values
import requests

from pathlib import Path

# Set directory path

In [ ]:
cwd = Path.cwd()
data_dir = Path(cwd / 'data')


# Connect to database on lightsail

In [ ]:
DB_HOST=os.environ.get('DB_HOST')
DB_NAME=os.environ.get('DB_NAME')
DB_USER=os.environ.get('DB_USER')
DB_PASSW=os.environ.get('DB_PASSWORD')
DB_PORT=os.environ.get('DB_PORT')

In [ ]:
#Connect to database citikike
try:
    conn = psycopg2.connect(
            host = DB_HOST,
            database = DB_NAME,
            user = DB_USER,
            password = DB_PASSW,
            port = DB_PORT,
            keepalives=1,
            keepalives_idle=30,
            keepalives_interval=10,
            keepalives_count=5
    )
    print('Connection Successful')

    cursor = conn.cursor()
except Exception as e:
    print(f'Could not establish connection:{e}')

This table has this constaintt start_station_id,end_station_id) 
It also has the following columns:

  - start_station_id 
  - end_station_id  
  - trip_count   
  - avg_duration_seconds   
  - member_trips   
  - casual_trips   
  - last_computed   

  Insert values

In [ ]:
cursor.execute('''
    SELECT * FROM station_pairs LIMIT 2;''')
cursor.fetchall()

In [ ]:
cursor.execute("""
    SELECT constraint_name, constraint_type
    FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS
    WHERE TABLE_NAME = 'station_pairs';
""")
cursor.fetchall()

#### Change FK constraint for start_station_id

In [ ]:
try:
    cursor.execute(
            """
            ALTER TABLE station_pairs 
            DROP CONSTRAINT station_pairs_start_station_id_fkey;
            """
       
    )
    conn.commit()
except Exceptoin as e:
    conn.rollback()
    print(f'Error: {e}')

#### Change FK constraint for end_station_id

In [ ]:
try:
    cursor.execute(
            """
            ALTER TABLE station_pairs 
            DROP CONSTRAINT station_pairs_end_station_id_fkey;
            """
    )
    conn.commit()
    print(f'Dropped constraint on end_station_id')
except Exceptoin as e:
    conn.rollback()
    print(f'Error: {e}')



In [ ]:
cursor.execute("""
    SELECT constraint_name, constraint_type
    FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS
    WHERE TABLE_NAME = 'station_pairs';
""")
cursor.fetchall()

### Insert into the table

In [ ]:
try:
    cursor.execute(
        '''
            INSERT INTO station_pairs(start_station_id, end_station_id, trip_count, avg_duration_seconds, member_trips, casual_trips, last_computed)
            SELECT 
                start_station_id,
                end_station_id,
                COUNT(*) AS trip_count,
                AVG(duration_seconds)::Integer AS avg_duration_seconds,
                COUNT(*) FILTER (WHERE member_casual = 'member' ) AS member_trips,
                COUNT(*) FILTER (WHERE member_casual = 'casual') AS casual_trips,
                CURRENT_TIMESTAMP AS last_computed
            FROM trips
            WHERE start_station_id IS NOT NULL AND end_station_id IS NOT NULL 
            GROUP BY start_station_id, end_station_id;
        ''')
    #commit
    conn.commit()
    print(f'There were {cursor.rowcount} rows inserted')
except Exception as e:
    conn.rollback()
    print(f'Error: {e}')

In [ ]:
cursor.execute('SELECT * FROM station_pairs LIMIT 2')
cursor.fetchall()

### check for nulls

In [ ]:
cursor.execute('''
   SELECT 
        COUNT(*) FILTER (WHERE trip_count = 1) as single_trips,
        COUNT(*) FILTER (WHERE trip_count = 2) as two_trips,
        COUNT(*) FILTER (WHERE trip_count BETWEEN 3 AND 5) as three_to_five,
        COUNT(*) FILTER (WHERE trip_count > 5) as more_than_five
    FROM station_pairs;
''')

cursor.fetchall()

In [ ]:
cursor.execute('''
SELECT 
    COUNT(*) FILTER (WHERE trip_count =1 ) AS short_trips,
    COUNT(*) FILTER (WHERE start_station_id = end_station_id),
    COUNT(*) FILTER (WHERE avg_duration_seconds IS NULL) AS null_duration
FROM station_pairs;
    ''')
cursor.fetchall()


In [ ]:
cursor.execute('DELETE FROM station_pairs WHERE start_station_id = end_station_id;')


In [ ]:
cursor.execute('DELETE FROM station_pairs WHERE trip_count = 0;')

In [ ]:
cursor.execute('DELETE FROM station_pairs WHERE avg_duration_seconds IS NULL;')

In [ ]:
cursor.execute('UPDATE station_pairs SET last_computed = CURRENT_TIMESTAMP;')


In [ ]:
conn.close()

- This is because start/end_station_id references stations.short_name not station_id
- I need to change this constraint

- Check constraints in stations table

In [ ]:
cursor.execute("SELECT column_name FROM information_schema.columns WHERE table_name = 'stations' ORDER BY ordinal_position;")
cursor.fetchall()

- print the count of distict short names in stations

In [ ]:
cursor.execute("""
SELECT COUNT(*) AS total_count_of_rows,
        COUNT(DISTINCT short_name) as num_distict_short_ids
FROM stations;""")
cursor.fetchall()

In [ ]:
cursor.execute("""
SELECT start_station_id
FROM trips
LIMIT 5;""")
cursor.fetchall()

- Check for duplicates, count of each short name should be 1

In [ ]:
cursor.execute('''
    SELECT short_name, COUNT(*)
    FROM stations
    GROUP BY short_name
    HAVING COUNT(*) > 1;
    ''')
cursor.fetchall()

In [ ]:
conn.close()